# **Imports**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.autograd as autograd
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, TensorDataset
import numpy as np
import copy

# **Configuration & Data Preperation**

In [2]:
BATCH_SIZE = 64
BUFFER_SIZE_PER_TASK = 50 # Το ιδανικό trade-off μας

# CIFAR-10: Κανονικοποίηση RGB στο [-1, 1] για την Tanh του Generator
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Κατέβασμα δεδομένων CIFAR-10...")
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

def get_task_data(dataset, classes):
    targets = np.array(dataset.targets)
    mask = np.isin(targets, classes)
    indices = np.where(mask)[0]
    return Subset(dataset, indices)

tasks = [[0, 1], [2, 3], [4, 5], [6, 7], [8, 9]]
train_loaders, test_loaders = [], []

for task_classes in tasks:
    train_loaders.append(DataLoader(get_task_data(train_dataset, task_classes), batch_size=BATCH_SIZE, shuffle=True))
    test_loaders.append(DataLoader(get_task_data(test_dataset, task_classes), batch_size=BATCH_SIZE, shuffle=False))

print("\n--- Έτοιμο το Split-CIFAR10! ---")

Κατέβασμα δεδομένων CIFAR-10...


100%|██████████| 170M/170M [00:03<00:00, 43.0MB/s]



--- Έτοιμο το Split-CIFAR10! ---


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Το μοντέλο θα τρέξει σε: {device}")

Το μοντέλο θα τρέξει σε: cuda


# **Tiny Replay Buffer**

In [4]:
class TinyReplayBuffer:
    def __init__(self, samples_per_task):
        self.samples_per_task = samples_per_task
        self.buffer_data = []
        self.buffer_targets = []

    def add_data(self, loader):
        all_data, all_targets = [], []
        for d, t in loader:
            all_data.append(d)
            all_targets.append(t)
        all_data = torch.cat(all_data)
        all_targets = torch.cat(all_targets)

        indices = torch.randperm(len(all_data))[:self.samples_per_task]
        self.buffer_data.append(all_data[indices])
        self.buffer_targets.append(all_targets[indices])

    def get_buffer_loader(self, batch_size):
        if len(self.buffer_data) == 0:
            return None
        b_data = torch.cat(self.buffer_data)
        b_targets = torch.cat(self.buffer_targets)
        return DataLoader(TensorDataset(b_data, b_targets), batch_size=batch_size, shuffle=True)

# **Architectures (VGG-Style CNN, WGAN-GP)**

In [5]:
class CNNClassifier(nn.Module):
    def __init__(self):
        super(CNNClassifier, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 8 * 8, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.5), nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.view(x.size(0), -1))

LATENT_DIM = 100
NUM_CLASSES = 10

class CondCNNGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, num_classes=NUM_CLASSES):
        super(CondCNNGenerator, self).__init__()
        self.label_emb = nn.Embedding(num_classes, 10)
        self.l1 = nn.Sequential(nn.Linear(latent_dim + 10, 256 * 4 * 4))
        self.conv_blocks = nn.Sequential(
            nn.InstanceNorm2d(256),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.InstanceNorm2d(128), nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.InstanceNorm2d(64), nn.LeakyReLU(0.2, inplace=True),
            nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1), nn.Tanh(),
        )

    def forward(self, noise, labels):
        c = self.label_emb(labels)
        gen_input = torch.cat((noise, c), dim=-1)
        out = self.l1(gen_input).view(gen_input.shape[0], 256, 4, 4)
        return self.conv_blocks(out)

class CondCNNCritic(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(CondCNNCritic, self).__init__()
        self.label_emb = nn.Embedding(num_classes, 32 * 32)
        self.conv_blocks = nn.Sequential(
            nn.Conv2d(4, 64, kernel_size=4, stride=2, padding=1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1), nn.LeakyReLU(0.2, inplace=True),
        )
        self.adv_layer = nn.Linear(256 * 4 * 4, 1)

    def forward(self, img, labels):
        c = self.label_emb(labels).view(labels.size(0), 1, 32, 32)
        x = torch.cat((img, c), dim=1)
        out = self.conv_blocks(x)
        return self.adv_layer(out.view(out.shape[0], -1))

def compute_gradient_penalty(critic, real_samples, fake_samples, labels):
    alpha = torch.rand(real_samples.size(0), 1, 1, 1).to(device)
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    d_interpolates = critic(interpolates, labels)
    fake = torch.ones(real_samples.size(0), 1).to(device)
    gradients = autograd.grad(outputs=d_interpolates, inputs=interpolates, grad_outputs=fake, create_graph=True, retain_graph=True, only_inputs=True)[0]
    return ((gradients.view(gradients.size(0), -1).norm(2, dim=1) - 1) ** 2).mean()

def evaluate_model(model, test_loaders, current_task_idx):
    model.eval()
    accuracies = []
    with torch.no_grad():
        for i in range(current_task_idx + 1):
            correct, total = 0, 0
            for data, target in test_loaders[i]:
                data, target = data.to(device), target.to(device)
                _, predicted = torch.max(model(data).data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()
            accuracies.append(100 * correct / total)
    return accuracies

# **Hybrid Training Loop (WGAN-GP + KD + Tiny Buffer)**

In [6]:
gr_classifier = CNNClassifier().to(device)
gr_G = CondCNNGenerator().to(device)
gr_C = CondCNNCritic().to(device)

opt_classifier = optim.Adam(gr_classifier.parameters(), lr=0.001)
opt_G = optim.Adam(gr_G.parameters(), lr=0.0001, betas=(0.0, 0.9))
opt_C = optim.Adam(gr_C.parameters(), lr=0.0001, betas=(0.0, 0.9))

criterion_class = nn.CrossEntropyLoss()
tiny_buffer = TinyReplayBuffer(samples_per_task=BUFFER_SIZE_PER_TASK)
prev_classifier, prev_G = None, None

EPOCHS_PER_TASK = 60
LAMBDA_GP = 10
CRITIC_ITERATIONS = 5
TEMPERATURE = 2.0

print(f"\n--- Ξεκινάει η Εκπαίδευση (CIFAR-10 | Hybrid WGAN-GP + KD | Buffer: {BUFFER_SIZE_PER_TASK} imgs/task) ---")

accuracy_matrix = []

for task_idx, train_loader in enumerate(train_loaders):
    print(f"\nΕκπαίδευση στο Task {task_idx + 1} (Κλάσεις {tasks[task_idx]})...")

    buffer_loader = tiny_buffer.get_buffer_loader(BATCH_SIZE)
    buffer_iter = iter(buffer_loader) if buffer_loader else None
    past_classes = [c for past_task in tasks[:task_idx] for c in past_task]

    for epoch in range(EPOCHS_PER_TASK):
        gr_classifier.train()
        gr_G.train()
        gr_C.train()

        for real_data, real_target in train_loader:
            real_data, real_target = real_data.to(device), real_target.to(device)
            batch_size = real_data.size(0)

            mem_data, mem_target = None, None
            if buffer_iter is not None:
                try:
                    mem_data, mem_target = next(buffer_iter)
                except StopIteration:
                    buffer_iter = iter(buffer_loader)
                    mem_data, mem_target = next(buffer_iter)
                mem_data, mem_target = mem_data.to(device), mem_target.to(device)

            real_mem_data = torch.cat([real_data, mem_data]) if mem_data is not None else real_data
            real_mem_target = torch.cat([real_target, mem_target]) if mem_target is not None else real_target

            fake_prev_data, fake_soft_targets, fake_prev_target_hard = None, None, None

            if prev_G is not None and prev_classifier is not None:
                fake_batch_size = batch_size * task_idx
                with torch.no_grad():
                    z_prev = torch.randn(fake_batch_size, LATENT_DIM).to(device)
                    y_prev_fake = torch.tensor(np.random.choice(past_classes, fake_batch_size)).to(device)
                    fake_prev_data = prev_G(z_prev, y_prev_fake).detach()

                    fake_logits = prev_classifier(fake_prev_data)
                    fake_soft_targets = F.softmax(fake_logits / TEMPERATURE, dim=1)
                    _, fake_prev_target_hard = torch.max(fake_logits.data, 1)

                combined_data = torch.cat([real_mem_data, fake_prev_data])
                combined_target = torch.cat([real_mem_target, fake_prev_target_hard])
            else:
                combined_data = real_mem_data
                combined_target = real_mem_target

            opt_classifier.zero_grad()
            out_real_mem = gr_classifier(real_mem_data)
            loss_class_real_mem = criterion_class(out_real_mem, real_mem_target)

            if fake_prev_data is not None:
                out_fake = gr_classifier(fake_prev_data)
                log_probs_fake = F.log_softmax(out_fake / TEMPERATURE, dim=1)
                loss_class_fake = F.kl_div(log_probs_fake, fake_soft_targets, reduction='batchmean') * (TEMPERATURE ** 2)
                loss_class = loss_class_real_mem + (task_idx * loss_class_fake)
            else:
                loss_class = loss_class_real_mem

            loss_class.backward()
            opt_classifier.step()

            total_batch_size = combined_data.size(0)

            for _ in range(CRITIC_ITERATIONS):
                opt_C.zero_grad()
                z_curr = torch.randn(total_batch_size, LATENT_DIM).to(device)
                fake_imgs = gr_G(z_curr, combined_target)

                real_validity = gr_C(combined_data, combined_target)
                fake_validity = gr_C(fake_imgs.detach(), combined_target)
                gp = compute_gradient_penalty(gr_C, combined_data, fake_imgs.detach(), combined_target)

                loss_C = -torch.mean(real_validity) + torch.mean(fake_validity) + LAMBDA_GP * gp
                loss_C.backward()
                opt_C.step()

            opt_G.zero_grad()
            z_curr = torch.randn(total_batch_size, LATENT_DIM).to(device)
            fake_imgs = gr_G(z_curr, combined_target)
            fake_validity = gr_C(fake_imgs, combined_target)

            loss_G = -torch.mean(fake_validity)
            loss_G.backward()
            opt_G.step()

    tiny_buffer.add_data(train_loader)

    accs = evaluate_model(gr_classifier, test_loaders, task_idx)
    accuracy_matrix.append(accs)
    print(f"Αποτελέσματα μετά το Task {task_idx + 1}:")
    for i, acc in enumerate(accs):
        print(f" -> Ακρίβεια στο Task {i + 1}: {acc:.2f}%")

    prev_classifier = copy.deepcopy(gr_classifier)
    prev_classifier.eval()
    prev_G = copy.deepcopy(gr_G)
    prev_G.eval()

N = len(accuracy_matrix)
forgetting = []
for j in range(N - 1):
    max_acc_past = max([accuracy_matrix[i][j] for i in range(j, N - 1)])
    final_acc = accuracy_matrix[N-1][j]
    forgetting.append(max_acc_past - final_acc)

avg_forgetting = sum(forgetting) / len(forgetting)
print("\n" + "="*50)
print(f"ΤΕΛΙΚΑ ΑΠΟΤΕΛΕΣΜΑΤΑ - Μέση Λήθη (Average Forgetting): {avg_forgetting:.2f}%")
print("="*50)


--- Ξεκινάει η Εκπαίδευση (CIFAR-10 | Hybrid WGAN-GP + KD | Buffer: 50 imgs/task) ---

Εκπαίδευση στο Task 1 (Κλάσεις [0, 1])...
Αποτελέσματα μετά το Task 1:
 -> Ακρίβεια στο Task 1: 97.05%

Εκπαίδευση στο Task 2 (Κλάσεις [2, 3])...
Αποτελέσματα μετά το Task 2:
 -> Ακρίβεια στο Task 1: 54.15%
 -> Ακρίβεια στο Task 2: 84.70%

Εκπαίδευση στο Task 3 (Κλάσεις [4, 5])...
Αποτελέσματα μετά το Task 3:
 -> Ακρίβεια στο Task 1: 37.50%
 -> Ακρίβεια στο Task 2: 3.50%
 -> Ακρίβεια στο Task 3: 87.20%

Εκπαίδευση στο Task 4 (Κλάσεις [6, 7])...
Αποτελέσματα μετά το Task 4:
 -> Ακρίβεια στο Task 1: 37.60%
 -> Ακρίβεια στο Task 2: 4.45%
 -> Ακρίβεια στο Task 3: 9.70%
 -> Ακρίβεια στο Task 4: 94.15%

Εκπαίδευση στο Task 5 (Κλάσεις [8, 9])...
Αποτελέσματα μετά το Task 5:
 -> Ακρίβεια στο Task 1: 3.10%
 -> Ακρίβεια στο Task 2: 10.50%
 -> Ακρίβεια στο Task 3: 18.45%
 -> Ακρίβεια στο Task 4: 55.35%
 -> Ακρίβεια στο Task 5: 92.10%

ΤΕΛΙΚΑ ΑΠΟΤΕΛΕΣΜΑΤΑ - Μέση Λήθη (Average Forgetting): 68.92%
